# Выдача рейтенга фильма

In [1]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


CUDA: True
NVIDIA GeForce RTX 5070 Ti


In [7]:
from pathlib import Path

ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
TRAIN_CSV = DATA_DIR / "scripts_ratingss.csv"
TXT_TRAIN_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"
MORE_TEST_TXT_DIR = DATA_DIR / "wrong"
print(ROOT)


C:\Users\Дмитрий\Downloads\Ratings


In [ ]:
%pip install -U transformers>=4.41
%pip install -U datasets peft accelerate bitsandbytes trl sentencepiece scikit-learn pandas

import accelerate


### Конфиг (параметры обучения)

In [9]:
import pandas as pd
import re
import os
from pathlib import Path
from typing import Dict, List, Optional

# ==================== КОНФИГУРАЦИЯ ПУТЕЙ ====================
ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
PROCESSED_DIR = DATA_DIR / "Pipe"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"
MORE_TEST_TXT_DIR = DATA_DIR / "wrong"

# Колонки в CSV (адаптировать под вашу структуру)
CSV_COLUMNS = {
    'filename': 'filename',      # Имя TXT файла (например: "brother_1997.txt")
    'title': 'title',            # Название фильма
    'year': 'year',              # Год выпуска
    'kp_rating': 'kp_rating',    # Рейтинг Кинопоиска
    'imdb_rating': 'imdb_rating',# Рейтинг IMDB
    'notes': 'notes',            # Дополнительные заметки/теги
    'age_rating_imdb': 'age_rating_imdb',  # Возрастной рейтинг IMDB
    'age_rating_kp': 'age_rating_kp',       # Возрастной рейтинг Кинопоиска
    'english_title':'english_title'
}
    
# Параметры обработки текста
MIN_TEXT_LENGTH = 1000  # Минимальная длина текста для обработки
MAX_TEXT_LENGTH = 260000  # Максимальная длина текста

# Кодировка файлов
ENCODING = 'utf-8'

# Создание директорий
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена. Директории созданы.")
print(f"ROOT: {ROOT}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

Конфигурация загружена. Директории созданы.
ROOT: C:\Users\Дмитрий\Downloads\Ratings
RAW_DATA_DIR: C:\Users\Дмитрий\Downloads\Ratings\datasets\scenaryy
PROCESSED_DIR: C:\Users\Дмитрий\Downloads\Ratings\datasets\Pipe


## Загрузка и проверка данных

In [10]:
class DataLoader:
    """Класс для загрузки и валидации данных"""
    
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        """Загрузка CSV с метаданными фильмов"""
        try:
            df = pd.read_csv(csv_path, encoding= ENCODING)
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            
            # Проверка необходимых колонок
            required_cols = list( CSV_COLUMNS.values())
            missing_cols = [col for col in required_cols if col not in df.columns]
            
            if missing_cols:
                raise ValueError(f"Отсутствуют колонки: {missing_cols}")
                
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        """Загрузка текста сценария из TXT файла"""
        try:
            with open(file_path, 'r', encoding= ENCODING) as f:
                text = f.read()
            
            if len(text) <  MIN_TEXT_LENGTH:
                warnings.warn(f"Файл {file_path.name} слишком короткий ({len(text)} символов)")
            
            return text
        except UnicodeDecodeError:
            # Попытка альтернативных кодировок
            for encoding in ['cp1251', 'iso-8859-1', 'mac_cyrillic']:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except:
                    continue
            raise Exception(f"Не удалось декодировать файл: {file_path.name}")
        except Exception as e:
            raise Exception(f"Ошибка чтения файла {file_path.name}: {e}")
    
    @staticmethod
    def validate_data_integrity(metadata_df: pd.DataFrame, raw_data_dir: Path) -> List[str]:
        """Проверка соответствия файлов в CSV и TXT"""
        txt_files = {f.stem for f in raw_data_dir.glob("*.txt")}
        csv_files = set(metadata_df[ CSV_COLUMNS['filename']].str.replace('.txt', ''))
        
        missing_in_csv = txt_files - csv_files
        missing_in_txt = csv_files - txt_files
        
        if missing_in_csv:
            print(f"Внимание: {len(missing_in_csv)} файлов в папке нет в CSV")
        
        if missing_in_txt:
            print(f"Внимание: {len(missing_in_txt)} файлов из CSV отсутствуют в папке")
        
        # Возвращаем список файлов, которые есть в обоих местах
        valid_files = list(txt_files.intersection(csv_files))
        print(f"Найдено {len(valid_files)} валидных файлов для обработки")
        
        return valid_files

# Загрузка данных
print("\n" + "="*50)
print("ЗАГРУЗКА ДАННЫХ")
print("="*50)

metadata_df = DataLoader.load_metadata( METADATA_CSV)
valid_filenames = DataLoader.validate_data_integrity(metadata_df,  RAW_DATA_DIR)

# Предпросмотр данных
print("\nПример данных из CSV:")
print(metadata_df.head(3))
print(f"\nКолонки: {list(metadata_df.columns)}")


ЗАГРУЗКА ДАННЫХ
Загружено 54 записей из scripts_ratingss.csv
Найдено 54 валидных файлов для обработки

Пример данных из CSV:
                                          filename  \
0                           8_миллиметров_Кино.txt   
1                        13_причин_почему_Кино.txt   
2  Kingsman_Секретная_служба_на_русском_читать.txt   

                                         title  year  kp_rating  imdb_rating  \
0                           8_миллиметров_Кино  1999        7.1          6.6   
1                        13_причин_почему_Кино  2017        7.3          7.4   
2  Kingsman_Секретная_служба_на_русском_читать  2015        7.7          7.7   

  notes age_rating_imdb age_rating_kp                  english_title  
0    ok               R           18+                   8 Millimeter  
1    ok           TV-MA     Not found           TH1RTEEN R3ASONS WHY  
2    ok       Not found           18+  Kingsman - The Secret Service  

Колонки: ['filename', 'title', 'year', 'kp_rating',

## Предобработка текста

In [11]:
class TextPreprocessor:
    """Класс для предобработки текста сценариев"""
    
    @staticmethod
    def clean_script_text(raw_text: str, 
                         remove_directions: bool = True,
                         remove_char_names: bool = False) -> str:
        """
        Очистка текста сценария
        """
        text = raw_text
        
        # 1. Нормализация переносов строк и пробелов
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)  # Удаление множественных переносов
        text = re.sub(r'[ \t]{2,}', ' ', text)  # Удаление множественных пробелов/табов
        
        # 2. Удаление ремарок и указаний сцены (опционально)
        if remove_directions:
            # Удаление текста в скобках (ремарки)
            text = re.sub(r'\([^)]*\)', '', text)
            # Удаление описаний сцен в верхнем регистре
            text = re.sub(r'^[A-ZА-Я\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        
        # 3. Удаление имен персонажей (опционально)
        if remove_char_names:
            # Паттерн: имя в верхнем регистре в начале строки
            text = re.sub(r'^[A-ZА-Я\s]+\n', '', text, flags=re.MULTILINE)
        
        # 4. Очистка от спецсимволов (сохраняем пунктуацию)
        text = re.sub(r'[^\w\s\.,!?;:\-\'\"\(\)\n]', '', text)
        
        # 5. Нормализация пробелов вокруг пунктуации
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        
        return text.strip()
    
    @staticmethod
    def extract_dialogues_only(text: str) -> str:
        """Извлечение только диалогов (базовый метод)"""
        lines = text.split('\n')
        dialogues = []
        
        for line in lines:
            # Удаляем строки, которые выглядят как ремарки или описания
            if (not line.strip().startswith('(') and 
                not line.strip().endswith(')') and
                len(line.strip()) > 0 and
                not line.strip().isupper()):
                dialogues.append(line.strip())
        
        return '\n'.join(dialogues)
    
    @staticmethod
    def calculate_text_metrics(text: str) -> Dict:
        """Расчет метрик текста"""
        words = text.split()
        sentences = re.split(r'[.!?]+', text)
        
        return {
            'total_chars': len(text),
            'total_words': len(words),
            'total_sentences': len([s for s in sentences if s.strip()]),
            'avg_word_length': sum(len(w) for w in words) / len(words) if words else 0,
            'unique_words': len(set(w.lower() for w in words))
        }

# Тестирование препроцессора
print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ ПРЕПРОЦЕССИНГА")
print("="*50)

# Пример с вашим файлом "Мистер_Робот_Пилот_Кино.txt"
test_file =  RAW_DATA_DIR / "Мистер_Робот_Пилот_Кино.txt"
if test_file.exists():
    test_text = DataLoader.load_script_txt(test_file)
    cleaned_text = TextPreprocessor.clean_script_text(test_text)
    metrics = TextPreprocessor.calculate_text_metrics(cleaned_text)
    
    print(f"Оригинальный текст: {len(test_text)} символов")
    print(f"Очищенный текст: {len(cleaned_text)} символов")
    print(f"Метрики: {metrics}")
    
    # Показать пример очищенного текста
    print("\nПример очищенного текста (первые 500 символов):")
    print(cleaned_text[:500] + "...")


ТЕСТИРОВАНИЕ ПРЕПРОЦЕССИНГА
Оригинальный текст: 59853 символов
Очищенный текст: 55106 символов
Метрики: {'total_chars': 55106, 'total_words': 8476, 'total_sentences': 1441, 'avg_word_length': 5.483836715431807, 'unique_words': 3842}

Пример очищенного текста (первые 500 символов):
Перевод: Диалоги  Григорий Сенин
Сценарий  Антон Базелинский

Привет, друг. Привет, друг? Какая глупость! Может, стоит дать тебе имя? Но это скользкая тропа. Ты только у меня в голове, нельзя об этом забывать. Черт! Это и вправду случилось. Я говорю с воображаемым человеком. Громкий, неистовый джаз СТАНОВИТСЯ ГРОМЧЕ на саундтреке. В черном кадре формируются силуэты. То, что я расскажу, совершенно секретно. Это заговор, который охватывает всех. Есть группа могущественных людей, которые тайно упра...


##  Создание структурированного датасета

In [12]:
class MovieDatasetBuilder:
    """Создание структурированного датасета фильмов"""
    
    @staticmethod
    def generate_filename_tags(row: pd.Series, metadata_df: pd.DataFrame) -> str:
        """
        Генерация имени файла с тегами по шаблону:
        Название_Год_РейтингKP_РейтингIMDB_ВозрастнойРейтинг_Теги.txt
        
        Пример: Брат_1997_8.2_7.6_18_Драма_Криминал.txt
        """
        # Базовое название (латиницей, без пробелов)
        title_clean = re.sub(r'[^\w]', '_', str(row[ CSV_COLUMNS['title']]))
        title_clean = title_clean[:50]  # Ограничение длины
        
        year = str(row[ CSV_COLUMNS['year']])[:4]
        
        # Рейтинги (форматирование)
        kp_rating = f"{float(row[ CSV_COLUMNS['kp_rating']]):.1f}" if pd.notna(row[ CSV_COLUMNS['kp_rating']]) else "0.0"
        imdb_rating = f"{float(row[ CSV_COLUMNS['imdb_rating']]):.1f}" if pd.notna(row[ CSV_COLUMNS['imdb_rating']]) else "0.0"
        
        # Возрастной рейтинг (берем максимальный из двух)
        age_kp = str(row[ CSV_COLUMNS['age_rating_kp']]) if pd.notna(row[ CSV_COLUMNS['age_rating_kp']]) else "NR"
        age_imdb = str(row[ CSV_COLUMNS['age_rating_imdb']]) if pd.notna(row[ CSV_COLUMNS['age_rating_imdb']]) else "NR"
        age_rating = age_kp if age_kp != "NR" else age_imdb
        
        # Извлечение тегов из notes (разделитель - запятая)
        notes = str(row[ CSV_COLUMNS['notes']]) if pd.notna(row[ CSV_COLUMNS['notes']]) else ""
        tags = []
        if notes:
            # Разделение по запятой, точке с запятой или пробелу
            for tag in re.split(r'[,;]', notes):
                tag_clean = tag.strip().replace(' ', '_')
                if tag_clean and len(tag_clean) < 20:
                    tags.append(tag_clean[:15])
        
        # Сборка имени файла
        filename_parts = [
            title_clean,
            year,
            f"KP{kp_rating}",
            f"IMDB{imdb_rating}",
            f"Age{age_rating}"
        ]
        
        # Добавление тегов (максимум 3 основных)
        filename_parts.extend(tags[:3])
        
        # Фильтрация пустых частей и объединение
        filename = "_".join([part for part in filename_parts if part]) + ".txt"
        
        # Удаление дублирующихся подчеркиваний
        filename = re.sub(r'_+', '_', filename)
        
        return filename
    
    @staticmethod
    def build_dataset(metadata_df: pd.DataFrame, 
                     raw_data_dir: Path,
                     processed_dir: Path) -> pd.DataFrame:
        """
        Основной метод построения датасета
        """
        processed_data = []
        
        for idx, row in metadata_df.iterrows():
            try:
                filename = str(row[ CSV_COLUMNS['filename']])
                if not filename.endswith('.txt'):
                    filename += '.txt'
                
                txt_path = raw_data_dir / filename
                
                if not txt_path.exists():
                    print(f"Файл не найден: {filename}")
                    continue
                
                # Загрузка и обработка текста
                raw_text = DataLoader.load_script_txt(txt_path)
                cleaned_text = TextPreprocessor.clean_script_text(raw_text)
                
                # Проверка минимальной длины
                if len(cleaned_text) <  MIN_TEXT_LENGTH:
                    print(f"Текст слишком короткий: {filename} ({len(cleaned_text)} chars)")
                    continue
                
                # Генерация нового имени файла с тегами
                new_filename = MovieDatasetBuilder.generate_filename_tags(row, metadata_df)
                
                # Сохранение обработанного файла
                processed_path = processed_dir / new_filename
                with open(processed_path, 'w', encoding= ENCODING) as f:
                    f.write(cleaned_text)
                
                # Расчет метрик
                metrics = TextPreprocessor.calculate_text_metrics(cleaned_text)
                
                # Добавление в датасет
                processed_data.append({
                    'original_filename': filename,
                    'processed_filename': new_filename,
                    'title': row[ CSV_COLUMNS['title']],
                    'year': row[ CSV_COLUMNS['year']],
                    'kp_rating': row[ CSV_COLUMNS['kp_rating']],
                    'imdb_rating': row[ CSV_COLUMNS['imdb_rating']],
                    'age_rating': row.get( CSV_COLUMNS['age_rating_kp'], 
                                         row.get( CSV_COLUMNS['age_rating_imdb'], 'NR')),
                    'tags': row[ CSV_COLUMNS['notes']] if pd.notna(row[ CSV_COLUMNS['notes']]) else '',
                    'text_length': metrics['total_chars'],
                    'word_count': metrics['total_words'],
                    'unique_words': metrics['unique_words'],
                    'processed_path': str(processed_path)
                })
                
                if idx % 10 == 0:
                    print(f"Обработано {idx+1}/{len(metadata_df)} файлов")
                    
            except Exception as e:
                print(f"Ошибка обработки файла {filename}: {e}")
                continue
        
        # Создание DataFrame
        processed_df = pd.DataFrame(processed_data)
        
        # Сохранение метаданных обработанных файлов
        metadata_output_path = processed_dir / "processed_metadata.csv"
        processed_df.to_csv(metadata_output_path, index=False, encoding= ENCODING)
        
        print(f"\nОбработка завершена!")
        print(f"Успешно обработано: {len(processed_df)} файлов")
        print(f"Метаданные сохранены: {metadata_output_path}")
        
        return processed_df

# Запуск обработки
print("\n" + "="*50)
print("ЗАПУСК ОБРАБОТКИ ВСЕХ ФАЙЛОВ")
print("="*50)

processed_df = MovieDatasetBuilder.build_dataset(
    metadata_df=metadata_df,
    raw_data_dir= RAW_DATA_DIR,
    processed_dir= PROCESSED_DIR
)

# Примеры созданных файлов
print("\nПримеры созданных файлов:")
if len(processed_df) > 0:
    for i, filename in enumerate(processed_df['processed_filename'].head(5)):
        print(f"{i+1}. {filename}")
    
    # Показать статистику
    print("\nСтатистика датасета:")
    print(f"Всего фильмов: {len(processed_df)}")
    print(f"Средняя длина текста: {processed_df['text_length'].mean():.0f} символов")
    print(f"Среднее количество слов: {processed_df['word_count'].mean():.0f}")
    
    # Пример для "Брата" (если есть в данных)
    if 'Брат' in processed_df['title'].values:
        brother_data = processed_df[processed_df['title'].str.contains('Брат')].iloc[0]
        print(f"\nПример для фильма 'Брат':")
        print(f"Оригинальный файл: {brother_data['original_filename']}")
        print(f"Обработанный файл: {brother_data['processed_filename']}")
        print(f"Теги в имени: {brother_data['tags']}")


ЗАПУСК ОБРАБОТКИ ВСЕХ ФАЙЛОВ
Обработано 1/54 файлов
Обработано 11/54 файлов
Обработано 21/54 файлов
Обработано 31/54 файлов
Обработано 41/54 файлов
Обработано 51/54 файлов

Обработка завершена!
Успешно обработано: 54 файлов
Метаданные сохранены: C:\Users\Дмитрий\Downloads\Ratings\datasets\Pipe\processed_metadata.csv

Примеры созданных файлов:
1. 8_миллиметров_Кино_1999_KP7.1_IMDB6.6_Age18+_ok.txt
2. 13_причин_почему_Кино_2017_KP7.3_IMDB7.4_AgeNot found_ok.txt
3. Kingsman_Секретная_служба_на_русском_читать_2015_KP7.7_IMDB7.7_Age18+_ok.txt
4. Большая_маленькая_ложь_Кино_2017_KP8.2_IMDB8.4_AgeNot found_ok.txt
5. Бриллиантовая_История_Кино_2024_KP7.3_IMDB6.3_AgeNot found_ok.txt

Статистика датасета:
Всего фильмов: 54
Средняя длина текста: 75117 символов
Среднее количество слов: 11573


## Подготовка к обучению

In [14]:
class ModelDataPreparer:
    """Подготовка данных для обучения модели"""
    
    @staticmethod
    def create_train_val_split(processed_df: pd.DataFrame, 
                             test_size: float = 0.2,
                             random_state: int = 42,
                             stratify_by: str = None):
        """Разделение на train/validation с защитой от редких классов"""
        from sklearn.model_selection import train_test_split
        
        # Проверяем возможность стратификации
        stratify = None
        if stratify_by and stratify_by in processed_df.columns:
            class_counts = processed_df[stratify_by].value_counts()
            print(f"Распределение классов в '{stratify_by}':")
            print(class_counts)
            
            # Проверяем, что в каждом классе минимум 2 образца
            if (class_counts >= 2).all():
                stratify = processed_df[stratify_by]
                print(f"✓ Стратификация по '{stratify_by}' активирована")
            else:
                rare_classes = class_counts[class_counts < 2].index.tolist()
                print(f"⚠ Некоторые классы слишком редкие для стратификации: {rare_classes}")
                print("  Разделение будет без стратификации")
        else:
            print(f"⚠ Колонка '{stratify_by}' не найдена или не указана")
        
        train_df, val_df = train_test_split(
            processed_df,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify
        )
        
        print(f"✓ Train: {len(train_df)} записей ({len(train_df)/len(processed_df)*100:.1f}%)")
        print(f"✓ Validation: {len(val_df)} записей ({len(val_df)/len(processed_df)*100:.1f}%)")
        
        return train_df, val_df
    
    @staticmethod
    def prepare_for_classification(processed_df: pd.DataFrame,
                                  target_column: str = 'kp_rating'):
        """Подготовка данных для классификации/регрессии"""
        
        if target_column not in processed_df.columns:
            print(f"✗ Колонка '{target_column}' не найдена в данных")
            print(f"  Доступные колонки: {list(processed_df.columns)}")
            return processed_df
        
        # Проверяем наличие пропущенных значений
        missing = processed_df[target_column].isnull().sum()
        if missing > 0:
            print(f"⚠ В колонке '{target_column}' {missing} пропущенных значений")
            # Заполняем медианой
            median_val = processed_df[target_column].median()
            processed_df[target_column] = processed_df[target_column].fillna(median_val)
            print(f"  Заполнено медианой: {median_val:.2f}")
        
        # Бинаризация рейтинга (пример для бинарной классификации)
        if target_column in ['kp_rating', 'imdb_rating']:
            # Создание бинарной метки (например, выше/ниже медианы)
            median_rating = processed_df[target_column].median()
            processed_df['label'] = (processed_df[target_column] > median_rating).astype(int)
            print(f"✓ Бинарная классификация: медиана = {median_rating:.2f}")
            
            # Анализ распределения
            label_counts = processed_df['label'].value_counts()
            print(f"  Распределение классов:")
            print(f"    Высокий рейтинг (1): {label_counts.get(1, 0)} записей")
            print(f"    Низкий рейтинг (0): {label_counts.get(0, 0)} записей")
            print(f"    Баланс: {label_counts.get(1, 0)/len(processed_df)*100:.1f}% / {label_counts.get(0, 0)/len(processed_df)*100:.1f}%")
        
        return processed_df
    
    @staticmethod
    def prepare_texts_for_llm(processed_df: pd.DataFrame,
                             prompt_template: str = None,
                             max_context_length: int = 500):
        """Форматирование текстов для LLM (аналогично вашему генератору анекдотов)"""
        
        if prompt_template is None:
            prompt_template = """Фильм: {title} ({year})
Рейтинг Кинопоиска: {kp_rating}
Рейтинг IMDB: {imdb_rating}
Возрастной рейтинг: {age_rating}
Теги: {tags}

Текст сценария:
{text}

Задача: Проанализировать сценарий и предсказать рейтинг.
Ответ:"""

        formatted_texts = []
        missing_paths = []
        
        for idx, row in processed_df.iterrows():
            try:
                # Проверяем наличие пути к обработанному файлу
                if 'processed_path' not in row or pd.isna(row['processed_path']):
                    missing_paths.append(idx)
                    continue
                
                # Загрузка обработанного текста
                file_path = Path(row['processed_path'])
                if not file_path.exists():
                    print(f"✗ Файл не найден: {file_path}")
                    missing_paths.append(idx)
                    continue
                
                with open(file_path, 'r', encoding='utf-8') as f:
                    text_content = f.read()
                
                # Ограничение длины для LLM
                if len(text_content) > max_context_length:
                    text_content = text_content[:max_context_length] + "..."
                
                # Форматирование промпта
                formatted = prompt_template.format(
                    title=row.get('title', 'Неизвестно'),
                    year=row.get('year', 'Неизвестно'),
                    kp_rating=row.get('kp_rating', 'Н/Д'),
                    imdb_rating=row.get('imdb_rating', 'Н/Д'),
                    age_rating=row.get('age_rating', 'Н/Д'),
                    tags=row.get('tags', 'Нет тегов'),
                    text=text_content
                )
                formatted_texts.append(formatted)
                
            except Exception as e:
                print(f"✗ Ошибка при обработке строки {idx}: {e}")
                continue
        
        if missing_paths:
            print(f"⚠ Пропущено {len(missing_paths)} записей из-за отсутствия путей к файлам")
        
        return formatted_texts

# Подготовка данных для модели
print("\n" + "="*50)
print("ПОДГОТОВКА ДЛЯ МОДЕЛИ")
print("="*50)

# 0. ПРОВЕРКА СТРУКТУРЫ ДАННЫХ
print("0. Анализ структуры данных:")
print(f"   Всего записей: {len(processed_df)}")
print(f"   Колонки: {list(processed_df.columns)}")
print(f"   Типы данных:\n{processed_df.dtypes}")

# Проверяем наличие age_rating
if 'age_rating' in processed_df.columns:
    print(f"\n   Анализ age_rating:")
    age_stats = processed_df['age_rating'].value_counts()
    print(age_stats)
    
    # Если есть редкие классы, предлагаем альтернативы
    rare_classes = age_stats[age_stats < 2].index.tolist()
    if rare_classes:
        print(f"   ⚠ Редкие классы (<2 записей): {rare_classes}")
        print("   Рекомендация: использовать стратификацию по 'label' после бинаризации")
else:
    print("   ⚠ Колонка 'age_rating' не найдена")

# 1. Разделение на train/val (без стратификации или с альтернативой)
print("\n1. Разделение данных:")
if 'age_rating' in processed_df.columns:
    # Проверяем, можно ли использовать стратификацию по age_rating
    train_df, val_df = ModelDataPreparer.create_train_val_split(
        processed_df,
        test_size=0.2,
        random_state=42,
        stratify_by='age_rating'  # Попробуем, но метод сам проверит
    )
else:
    # Если нет age_rating, разделяем без стратификации
    train_df, val_df = ModelDataPreparer.create_train_val_split(
        processed_df,
        test_size=0.2,
        random_state=42
    )

# 2. Подготовка для классификации
print("\n2. Подготовка для классификации:")
print("   Для тренировочных данных:")
train_df = ModelDataPreparer.prepare_for_classification(train_df, target_column='kp_rating')

print("\n   Для валидационных данных:")
val_df = ModelDataPreparer.prepare_for_classification(val_df, target_column='kp_rating')

# 3. Форматирование для LLM (аналогично вашему генератору анекдотов)
print("\n3. Форматирование для LLM:")
train_texts = ModelDataPreparer.prepare_texts_for_llm(train_df)
val_texts = ModelDataPreparer.prepare_texts_for_llm(val_df)

print(f"\n✓ Создано {len(train_texts)} тренировочных промптов")
print(f"✓ Создано {len(val_texts)} валидационных промптов")

# Проверяем баланс классов
if 'label' in train_df.columns:
    print(f"\nБаланс классов в train: {train_df['label'].value_counts().to_dict()}")
if 'label' in val_df.columns:
    print(f"Баланс классов в val: {val_df['label'].value_counts().to_dict()}")

# Сохраняем примеры промптов
if train_texts:
    print(f"\nПример промпта (первые 500 символов):")
    print("-" * 50)
    print(train_texts[0][:500] + "...")
    
    # Сохраняем пример в файл
    sample_path = Path("example_prompt.txt")
    with open(sample_path, 'w', encoding='utf-8') as f:
        f.write(train_texts[0])
    print(f"\n✓ Пример промпта сохранен в: {sample_path}")

# АЛЬТЕРНАТИВНЫЙ ВАРИАНТ: РАЗДЕЛЕНИЕ С СТРАТИФИКАЦИЕЙ ПО ЛЕЙБЛАМ
print("\n" + "="*50)
print("АЛЬТЕРНАТИВНЫЙ ВАРИАНТ: Сначала бинаризация, потом стратификация")
print("="*50)

# Этот вариант лучше сохраняет баланс классов
def alternative_split_strategy(processed_df):
    """Альтернативная стратегия: сначала создаем лейблы, потом стратифицируем"""
    
    # 1. Создаем лейблы для всего датасета
    temp_df = processed_df.copy()
    if 'kp_rating' in temp_df.columns:
        median_rating = temp_df['kp_rating'].median()
        temp_df['label'] = (temp_df['kp_rating'] > median_rating).astype(int)
        print(f"Создан бинарный лейбл (медиана: {median_rating:.2f})")
        
        # 2. Разделяем с стратификацией по лейблу
        train_df, val_df = ModelDataPreparer.create_train_val_split(
            temp_df,
            test_size=0.2,
            random_state=42,
            stratify_by='label'
        )
        
        return train_df, val_df
    else:
        print("✗ Невозможно создать лейблы: колонка 'kp_rating' не найдена")
        return None, None

# Пример использования альтернативной стратегии
train_df_alt, val_df_alt = alternative_split_strategy(processed_df)

if train_df_alt is not None:
    print("\nРезультаты альтернативного разделения:")
    print(f"Train размер: {len(train_df_alt)}")
    print(f"Val размер: {len(val_df_alt)}")
    
    if 'label' in train_df_alt.columns:
        print(f"Распределение в train: {train_df_alt['label'].value_counts().to_dict()}")
        print(f"Распределение в val: {val_df_alt['label'].value_counts().to_dict()}")

# 4. СОХРАНЕНИЕ ПРОМПТОВ В ФАЙЛЫ
print("\n" + "="*50)
print("СОХРАНЕНИЕ ДАННЫХ ДЛЯ ОБУЧЕНИЯ")
print("="*50)

# Создаем директорию для промптов
prompts_dir = Path("prompts")
prompts_dir.mkdir(exist_ok=True)

# Сохраняем промпты
train_prompts_path = prompts_dir / "train_prompts.txt"
val_prompts_path = prompts_dir / "val_prompts.txt"

with open(train_prompts_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(train_texts))

with open(val_prompts_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(val_texts))

print(f"✓ Тренировочные промпты сохранены: {train_prompts_path}")
print(f"✓ Валидационные промпты сохранены: {val_prompts_path}")

# Сохраняем метаданные разделения
train_df.to_csv(prompts_dir / "train_metadata.csv", index=False, encoding='utf-8')
val_df.to_csv(prompts_dir / "val_metadata.csv", index=False, encoding='utf-8')
print(f"✓ Метаданные разделения сохранены")

# 5. СТАТИСТИКА ПО ФАЙЛАМ
print("\n" + "="*50)
print("СТАТИСТИКА ОБРАБОТКИ")
print("="*50)

print(f"Исходный датасет: {len(processed_df)} фильмов")
print(f"Тренировочный набор: {len(train_df)} фильмов ({len(train_df)/len(processed_df)*100:.1f}%)")
print(f"Валидационный набор: {len(val_df)} фильмов ({len(val_df)/len(processed_df)*100:.1f}%)")
print(f"Всего промптов создано: {len(train_texts) + len(val_texts)}")

if 'label' in train_df.columns:
    train_pos = train_df['label'].sum()
    train_neg = len(train_df) - train_pos
    print(f"\nКлассы в train: {train_pos} положительных, {train_neg} отрицательных")
    
if 'label' in val_df.columns:
    val_pos = val_df['label'].sum()
    val_neg = len(val_df) - val_pos
    print(f"Классы в val: {val_pos} положительных, {val_neg} отрицательных")

print("\n✓ Все этапы подготовки данных завершены успешно!")


ПОДГОТОВКА ДЛЯ МОДЕЛИ
0. Анализ структуры данных:
   Всего записей: 54
   Колонки: ['original_filename', 'processed_filename', 'title', 'year', 'kp_rating', 'imdb_rating', 'age_rating', 'tags', 'text_length', 'word_count', 'unique_words', 'processed_path']
   Типы данных:
original_filename      object
processed_filename     object
title                  object
year                    int64
kp_rating             float64
imdb_rating           float64
age_rating             object
tags                   object
text_length             int64
word_count              int64
unique_words            int64
processed_path         object
dtype: object

   Анализ age_rating:
age_rating
Not found    40
18+          12
0+            1
12+           1
Name: count, dtype: int64
   ⚠ Редкие классы (<2 записей): ['0+', '12+']
   Рекомендация: использовать стратификацию по 'label' после бинаризации

1. Разделение данных:
Распределение классов в 'age_rating':
age_rating
Not found    40
18+          12
0+  

## Результаты 

In [15]:
# Сохранение финальных данных для обучения
print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ЭКСПОРТ")
print("="*50)

# Сохранение train/val split
train_df.to_csv( PROCESSED_DIR / "train_dataset.csv", index=False, encoding= ENCODING)
val_df.to_csv( PROCESSED_DIR / "val_dataset.csv", index=False, encoding= ENCODING)

# Сохранение форматированных текстов
with open( PROCESSED_DIR / "train_prompts.txt", 'w', encoding= ENCODING) as f:
    f.write('\n'.join(train_texts))

with open( PROCESSED_DIR / "val_prompts.txt", 'w', encoding= ENCODING) as f:
    f.write('\n'.join(val_texts))

# Создание README файла
readme_content = f"""
ПАЙПЛАЙН ПРЕДОБРАБОТКИ ФИЛЬМОВ
=============================

Структура проекта:
{ ROOT}/
├── raw_data/                    # Исходные TXT файлы
├── metadata/                    # CSV с метаданными
├── processed_data/              # Обработанные файлы
│   ├── [фильм_с_тегами].txt    # Обработанные тексты
│   ├── processed_metadata.csv   # Метаданные обработанных файлов
│   ├── train_dataset.csv        # Тренировочные данные
│   ├── val_dataset.csv          # Валидационные данные
│   ├── train_prompts.txt        # Промпты для обучения
│   └── val_prompts.txt          # Промпты для валидации
└── outputs/                     # Результаты моделей

Статистика обработки:
- Всего обработано файлов: {len(processed_df)}
- Тренировочная выборка: {len(train_df)}
- Валидационная выборка: {len(val_df)}

Формат имен файлов:
Название_Год_РейтингKP_РейтингIMDB_ВозрастнойРейтинг_Тег1_Тег2.txt

Пример: Брат_1997_KP8.2_IMDB7.6_Age18_Драма_Криминал.txt

Дата обработки: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
"""

with open( ROOT / "README.md", 'w', encoding= ENCODING) as f:
    f.write(readme_content)

print("Обработка завершена успешно!")
print(f"Все файлы сохранены в: { PROCESSED_DIR}")
print(f"README файл создан: { ROOT / 'README.md'}")


ФИНАЛЬНЫЙ ЭКСПОРТ
Обработка завершена успешно!
Все файлы сохранены в: C:\Users\Дмитрий\Downloads\Ratings\datasets\Pipe
README файл создан: C:\Users\Дмитрий\Downloads\Ratings\README.md


# DL

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    force_download=True,
)
print("Tokenizer loaded OK")


### Baseline

In [ ]:
import torch
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    force_download=True, 
)
print("Model loaded OK")


def generate(model, prompt: str, max_new_tokens=160, temperature=0.9, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.08,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

baseline_prompts = [
    "Расскажи мне анекдот про программиста.",
    "Расскажи мне анекдот про врача и пациента.",
    "Расскажи мне анекдот про студента на экзамене.",
    "Расскажи мне анекдот про кота.",
    "Расскажи мне анекдот про тёщу.",
    "Расскажи мне анекдот про работу в офисе.",
    "Расскажи мне анекдот про школу.",
    "Расскажи мне анекдот про путешествия.",
]

baseline_outputs = [generate(model, p) for p in baseline_prompts]

for p, a in zip(baseline_prompts, baseline_outputs):
    print("="*90)
    print("PROMPT:", p)
    print("BASELINE:", a)


In [ ]:
import pandas as pd
df_baseline = pd.DataFrame({"prompt": baseline_prompts, "baseline": baseline_outputs})
df_baseline.to_csv(BASELINE_CSV, index=False, encoding="utf-8")
print("Saved baseline:", BASELINE_CSV)
df_baseline.head(3)


### Train/Val split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df_h[["sft_text"]], test_size=0.02, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

len(train_df), len(val_df)


### LoRA + настройка модели

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb,
)

base_4bit = prepare_model_for_kbit_training(base_4bit)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(base_4bit, lora_cfg)
model.print_trainable_parameters()


### Токенизация

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

def tokenize(batch):
    tok = tokenizer(
        batch["sft_text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    tok["labels"] = tok["input_ids"].copy()
    tok["length"] = [len(x) for x in tok["input_ids"]]
    return tok


train_tok = train_ds.map(tokenize, batched=True, remove_columns=["sft_text"])
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=["sft_text"])


train_tok[0].keys(), len(train_tok), len(val_tok)


### Обучалка

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "qwen25_15b_lora_run"),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACC_STEPS,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    warmup_ratio=0.03,

    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    fp16=True,
    report_to="none",
)



collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    data_collator=collator,
)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
trainer.train()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("Saved adapter:", ADAPTER_DIR)


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_reload = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb,
)
ft = PeftModel.from_pretrained(base_reload, str(ADAPTER_DIR)).eval()

def gen_ft(prompt: str, max_new_tokens=260):
    inputs = tokenizer(prompt, return_tensors="pt").to(ft.device)
    with torch.no_grad():
        out = ft.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            repetition_penalty=1.08,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(gen_ft("Расскажи мне анекдот про программиста."))
